# Library import

In [ ]:
import pandas as pd
import numpy as np
import gc
from datetime import datetime as dt
from datetime import date
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings("ignore")

import hydroeval as he
from lumod import tools, MonteCarlo
from lumod.models import HBV

In [3]:
def get_hydro_data_on_index(index_hydro, hydro_data):
    hydro_data = hydro_data.loc[hydro_data["index"] == index_hydro]
    hydro_data['precip_amount_org'] = hydro_data['precip_amount_org'].fillna(0)
    hydro_data.set_index("date", inplace=True)
    hydro_data = hydro_data.rename(columns={"q": "qt", "avg_air_temp_org": "tmean", "precip_amount_org": "prec"})
    hydro_data = hydro_data[["prec", "tmean", "qt", "pet"]]
    return hydro_data

In [4]:
def get_post_info(df, index):
    df = df.loc[df["ids"] == index]
    area = float(df['Площ'])
    lat = float(df['lat'])
    return area, lat

In [5]:
hydro_meteo_data = pd.read_excel("data/qtpet_pairs_2.xlsx")
hydro_post_data = pd.read_excel("data/hydrology_stations.xlsx")

In [45]:
index_hydro = 11117

In [46]:
hydro_data = get_hydro_data_on_index(index_hydro, hydro_meteo_data)

In [47]:
hydro_data['2011-01-01':'2021-12-31']

,prec,tmean,qt,pet
date,,,,
2011-01-01,0.6,-26.4,8.01,0.0
2011-01-02,0.0,-35.7,7.76,0.0
2011-01-03,0.0,-38.0,7.51,0.0
2011-01-04,0.0,-37.2,7.26,0.0
2011-01-05,0.0,-37.7,7.01,0.0
...,...,...,...,...
2021-12-27,0.0,-19.5,6.48,0.0
2021-12-28,0.0,-19.4,6.46,0.0
2021-12-29,0.0,-22.7,6.44,0.0


# функция калибровки

In [48]:
area, lat = get_post_info(hydro_post_data, index_hydro)

In [49]:
model_HBV = HBV(
    area = area,
    lat = lat,
)
xobs = hydro_data[["qt"]]

bounds =  { 
    "tthres": (-2, 5),
    "dd": (1, 6),
    "beta": (1, 6),
    "fc": (0, 1000),
    "k0": (0.01, 9),
    "k1": (0.001, 2),
    "k2": (0.0001, 0.01),
    "kp": (0, 2), # recession coefficient of percolation (1/d)
    "snow0": (-10, 100),
    "w01": (10, 900),
    "w02": (10, 300),
}
score1 = {
    "var": "qt",
    "metric": "nse",
    "weight": 1
}
scores = [score1]

mc_res = MonteCarlo(
    model_HBV,
    hydro_data['2011-01-01':'2021-12-31'],
    bounds,
    numsimul=250000,
    save_vars=["qt"],
    xobs=xobs,
    scores=scores,
    keep_best=1,
    start='2011-01-01',
    end='2021-12-31'
)
gc.collect()

Progress: |██████████████████████████████████████████████████| 100.0% Complete


0

In [50]:
parameters = mc_res["parameters"].to_dict(orient="index")[1]

In [51]:
parameters

{'maxbas': 3.0,
 'tthres': 0.9158437252044678,
 'dd': 2.083845853805542,
 'cevp': 2.0,
 'cevpam': 1.0,
 'cevpph': 0.0,
 'beta': 1.1751477718353271,
 'fc': 4.757287502288818,
 'pwp': 0.800000011920929,
 'k0': 6.832691192626953,
 'k1': 0.0841885358095169,
 'k2': 0.004411450121551752,
 'kp': 1.398065447807312,
 'lthres': 50.0,
 'snow0': 87.41828918457031,
 's0': 0.5,
 'w01': 48.809879302978516,
 'w02': 180.8386993408203}

In [52]:
def calibration(df, start_c, end_c, area, lat, parameters_calibration):
    model_HBV = HBV(area=area, lat=lat, params=parameters_calibration)

    sim_cal = model_HBV.run(df, start=start_c, end=end_c)
    metrics_cal = tools.metrics.summary(df["qt"], sim_cal["qt"])
    print("Calibration period")
    
    tools.plots.model_evaluation(df["prec"], df["qt"], sim_cal["qt"],
                                 start=start_c, end=end_c)

    return metrics_cal, sim_cal

In [ ]:
metrics, cal= calibration(hydro_data, "2011-01-01", "2021-12-31", area, lat, parameters)

In [54]:
cal_date_start_minus_one = date(2011, 1, 1)
cal_date_end = date(2021, 12, 31)

In [55]:
df_hydro_calibration = pd.DataFrame()
df_hydro_calibration["Observed values"] =  hydro_data.loc[cal_date_start_minus_one:cal_date_end].qt
df_hydro_calibration["Simulated values"] =  cal.loc[cal_date_start_minus_one:cal_date_end].qt
df_hydro_calibration = df_hydro_calibration[cal_date_start_minus_one.replace(year=cal_date_start_minus_one.year + 1):]
df_hydro_calibration = df_hydro_calibration.dropna()
df_hydro_calibration = df_hydro_calibration[df_hydro_calibration["Simulated values"] != np.inf]
df_hydro_calibration = df_hydro_calibration[df_hydro_calibration["Simulated values"] > -np.inf]

In [56]:
df_hydro_calibration.to_excel(f"result/{hydro_post_data.loc[hydro_post_data["ids"] == index_hydro]["name"].values[0]}({index_hydro}) NSE MC v1. 2012 по 2021.xlsx")

In [ ]:

metrics_calibration = {
    "NSE" : round(float((he.evaluator(he.nse, df_hydro_calibration["Simulated values"].values, df_hydro_calibration["Observed values"].values))[0]), 2), 
    "KGE": round(float((he.evaluator(he.kge, df_hydro_calibration["Simulated values"].values, df_hydro_calibration["Observed values"].values))[0]), 2),
    "RMSE": round(root_mean_squared_error(df_hydro_calibration["Observed values"].values, df_hydro_calibration["Simulated values"].values), 2),
    "MAE": round(mean_absolute_error(df_hydro_calibration["Observed values"].values, df_hydro_calibration["Simulated values"].values), 2),
}
metrics_calibration = pd.DataFrame.from_dict([metrics_calibration])
metrics_calibration

In [ ]:
numeric_cols = ['Observed values', 'Simulated values']
years = sorted(df_hydro_calibration.index.year.unique())

# Оптимальное количество колонок
ncols = 2
nrows = (len(years) + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows=nrows, ncols=ncols, 
    figsize=(20, nrows*2.5), 
    constrained_layout=True
)

# fig.suptitle('Калибровка ЭКОМАГ', fontsize=16, y=1.02)
date_format = mdates.DateFormatter("%b")

# Заглушки для легенды
lines = []
for col in numeric_cols:
    line, = axes.flat[0].plot([], [], label=col)
    lines.append(line)

# Отрисовка графиков
for i, year in enumerate(years):
    ax = axes.flat[i]
    year_data = df_hydro_calibration[df_hydro_calibration.index.year == year]

    ax.plot(year_data['Observed values'], color="#1f77b4", linewidth=1.3)
    ax.plot(year_data['Simulated values'], color="#d62728", linewidth=1.3)
    ax.set_ylabel("Q, m³/s", fontsize=9)
    ax.set_title(f'{year} year', fontsize=9)
    ax.xaxis.set_major_formatter(date_format)
    ax.grid(True, linestyle='--', alpha=0.4)

# Убираем пустые оси
for j in range(i+1, nrows*ncols):
    fig.delaxes(axes.flat[j])

# Легенду переносим вниз
# fig.legend(
#     handles=lines,
#     labels=numeric_cols,
#     loc='upper left',
#     ncol=2,
#     frameon=False,
#     fontsize=12
# )
# Добавляем отступ снизу под легенду
plt.subplots_adjust(bottom=0.10)

# plt.savefig("annual_comparison.pdf", dpi=300, bbox_inches="tight")
plt.show()